In [6]:
import sys
print(sys.executable)
!{sys.executable} -m pip install xgboost

/opt/anaconda3/bin/python
  Using cached xgboost-3.3.0-py3-none-macosx_12_0_arm64.whl.metadata (2.0 kB)
Using cached xgboost-3.3.0-py3-none-macosx_12_0_arm64.whl (2.4 MB)


In [7]:
# =========================
# 1. Import thư viện
# =========================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    median_absolute_error,
    mean_absolute_percentage_error
)

from xgboost import XGBRegressor

In [9]:
# =========================
# 2. Đọc dữ liệu Zillow Final
# =========================

df = pd.read_csv("zillow_final.csv")

df.head()

,price,bed,bath,living,lot_sqft,lot_living,bed_bath,type_condo,type_manufactured,type_multi,...,risk_overall,risk_loss,risk_social,risk_resilience,risk_fire,risk_earthquake,risk_heat,dist_city,dist_airport,dist_coast
0,4980000.0,4.0,5.0,4126.0,4922.000,1.192923,0.8,False,False,False,...,80.718966,89.710407,9.209856,12.692809,73.920540,94.138632,8.354783,13.715619,17.888756,13.623171
1,1215000.0,3.0,2.0,1825.0,7840.800,4.296329,1.5,False,False,False,...,75.714982,83.604998,23.062252,12.692809,18.467649,89.615069,13.904381,39.379252,34.192540,22.666476
2,2629000.0,4.0,4.0,3019.0,43381.404,14.369461,1.0,False,False,False,...,92.322785,96.534514,9.209856,12.692809,98.160370,93.614213,13.158396,18.768005,19.185657,11.646300
3,400000.0,3.0,2.0,944.0,4268.880,4.522119,1.5,False,False,True,...,68.423055,43.952134,93.082983,12.692809,0.000000,93.975717,20.717146,9.473542,11.193127,18.586372
4,849000.0,2.0,2.0,1154.0,5002.000,4.334489,1.0,False,False,False,...,53.028195,54.870000,40.578096,12.692809,0.000000,87.909814,8.907717,37.911992,27.545766,35.124601


In [10]:
# =========================
# 3. Kiểm tra nhanh dữ liệu
# =========================

print("Shape:", df.shape)
print(df.info())
print(df.isnull().sum())
print("Duplicated:", df.duplicated().sum())

Shape: (4468, 33)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4468 entries, 0 to 4467
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   price              4468 non-null   float64
 1   bed                4468 non-null   float64
 2   bath               4468 non-null   float64
 3   living             4468 non-null   float64
 4   lot_sqft           4468 non-null   float64
 5   lot_living         4468 non-null   float64
 6   bed_bath           4468 non-null   float64
 7   type_condo         4468 non-null   bool   
 8   type_manufactured  4468 non-null   bool   
 9   type_multi         4468 non-null   bool   
 10  type_single        4468 non-null   bool   
 11  type_townhouse     4468 non-null   bool   
 12  lat                4468 non-null   float64
 13  long               4468 non-null   float64
 14  income             4468 non-null   float64
 15  poverty            4468 non-null   float64
 16  unempl

In [11]:
# =========================
# 4. Tách X và y
# Không get_dummies nữa vì file đã xử lý sẵn type rồi
# =========================

X = df.drop("price", axis=1)
y = df["price"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()

X shape: (4468, 32)
y shape: (4468,)


,bed,bath,living,lot_sqft,lot_living,bed_bath,type_condo,type_manufactured,type_multi,type_single,...,risk_overall,risk_loss,risk_social,risk_resilience,risk_fire,risk_earthquake,risk_heat,dist_city,dist_airport,dist_coast
0,4.0,5.0,4126.0,4922.000,1.192923,0.8,False,False,False,True,...,80.718966,89.710407,9.209856,12.692809,73.920540,94.138632,8.354783,13.715619,17.888756,13.623171
1,3.0,2.0,1825.0,7840.800,4.296329,1.5,False,False,False,True,...,75.714982,83.604998,23.062252,12.692809,18.467649,89.615069,13.904381,39.379252,34.192540,22.666476
2,4.0,4.0,3019.0,43381.404,14.369461,1.0,False,False,False,True,...,92.322785,96.534514,9.209856,12.692809,98.160370,93.614213,13.158396,18.768005,19.185657,11.646300
3,3.0,2.0,944.0,4268.880,4.522119,1.5,False,False,True,False,...,68.423055,43.952134,93.082983,12.692809,0.000000,93.975717,20.717146,9.473542,11.193127,18.586372
4,2.0,2.0,1154.0,5002.000,4.334489,1.0,False,False,False,True,...,53.028195,54.870000,40.578096,12.692809,0.000000,87.909814,8.907717,37.911992,27.545766,35.124601


In [12]:
# =========================
# 5. Nếu có cột True/False thì đổi thành 0/1
# Cái này không phải xử lý phức tạp, chỉ để model đọc chắc chắn hơn
# =========================

for col in X.select_dtypes(include=["bool"]).columns:
    X[col] = X[col].astype(int)

X.head()

,bed,bath,living,lot_sqft,lot_living,bed_bath,type_condo,type_manufactured,type_multi,type_single,...,risk_overall,risk_loss,risk_social,risk_resilience,risk_fire,risk_earthquake,risk_heat,dist_city,dist_airport,dist_coast
0,4.0,5.0,4126.0,4922.000,1.192923,0.8,0,0,0,1,...,80.718966,89.710407,9.209856,12.692809,73.920540,94.138632,8.354783,13.715619,17.888756,13.623171
1,3.0,2.0,1825.0,7840.800,4.296329,1.5,0,0,0,1,...,75.714982,83.604998,23.062252,12.692809,18.467649,89.615069,13.904381,39.379252,34.192540,22.666476
2,4.0,4.0,3019.0,43381.404,14.369461,1.0,0,0,0,1,...,92.322785,96.534514,9.209856,12.692809,98.160370,93.614213,13.158396,18.768005,19.185657,11.646300
3,3.0,2.0,944.0,4268.880,4.522119,1.5,0,0,1,0,...,68.423055,43.952134,93.082983,12.692809,0.000000,93.975717,20.717146,9.473542,11.193127,18.586372
4,2.0,2.0,1154.0,5002.000,4.334489,1.0,0,0,0,1,...,53.028195,54.870000,40.578096,12.692809,0.000000,87.909814,8.907717,37.911992,27.545766,35.124601


In [13]:
# =========================
# 6. Chia train/test
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(3574, 32)
(894, 32)


In [14]:
# =========================
# 7. Hàm đánh giá mô hình
# =========================

def evaluate_model(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    median_ae = median_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    print("=" * 55)
    print(model_name)
    print("=" * 55)
    print(f"MAE       : {mae:,.0f}")
    print(f"Median AE : {median_ae:,.0f}")
    print(f"RMSE      : {rmse:,.0f}")
    print(f"R2        : {r2:.4f}")
    print(f"MAPE      : {mape:.2f}%")
    print()

In [15]:
# =========================
# Model 1: Linear Regression
# =========================

linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

y_pred_linear = linear_model.predict(X_test)

evaluate_model(
    "Linear Regression - Raw Price",
    y_test,
    y_pred_linear
)

Linear Regression - Raw Price
MAE       : 2,245,498
Median AE : 1,383,245
RMSE      : 4,471,770
R2        : 0.6424
MAPE      : 207.93%



In [16]:
# =========================
# Model 2: Random Forest
# =========================

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

evaluate_model(
    "Random Forest - Raw Price",
    y_test,
    y_pred_rf
)

Random Forest - Raw Price
MAE       : 942,818
Median AE : 156,339
RMSE      : 3,816,223
R2        : 0.7396
MAPE      : 28.76%



In [17]:
# =========================
# Model 3: XGBoost
# =========================

xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

evaluate_model(
    "XGBoost - Raw Price",
    y_test,
    y_pred_xgb
)

XGBoost - Raw Price
MAE       : 981,617
Median AE : 173,175
RMSE      : 4,493,445
R2        : 0.6389
MAPE      : 30.38%

